# Multi-Agent Reinforcement Learning PettingZoo

In [1]:
from environment import TrafficEnvironment

from keychain import Keychain as kc
import os
from services import Trainer
from services import create_agent_objects
from services import confirm_env_variable
from services import get_json
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_checker import check_env


import ray
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv
from ray.tune.registry import register_env

from pettingzoo.test import parallel_api_test

from gymnasium.spaces import Box, Discrete
from ray.rllib.algorithms.dqn import DQNConfig
from ray.rllib.algorithms.dqn.dqn_torch_model import DQNTorchModel
from ray.rllib.env import PettingZooEnv
from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.fcnet import FullyConnectedNetwork as TorchFC
from ray.rllib.utils.framework import try_import_torch
from ray.rllib.utils.torch_utils import FLOAT_MAX
from ray.tune.registry import register_env

from pettingzoo.classic import leduc_holdem_v4

from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.torch_modelv2 import TorchModelV2
from ray.tune.registry import register_env
from torch import nn
import gymnasium as gym


confirm_env_variable(kc.SUMO_HOME, append="tools")
params = get_json(kc.PARAMS_PATH)

[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: C:\Program Files (x86)\Eclipse\Sumo\tools
[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: C:\Program Files (x86)\Eclipse\Sumo\tools


###  Initialization of the TrafficEnvironment 

- **Network**: initialize network, create demand, create paths, calculate free flow travel times.

- **Learning**: initialize observation and action space.

In [2]:
env = TrafficEnvironment(params[kc.SIMULATION_PARAMETERS])

[SUCCESS] Generated & saved 6 paths to: paths.csv
[SUCCESS] Environment initiated!


### Parallel API Test 

To make sure that our custom environment is consistent with the API: [blue_text](https://pettingzoo.farama.org/content/environment_tests/)

In [3]:
parallel_api_test(env, num_cycles=1_000_000)

joint_action is:  {'agent1': 1, 'agent2': 1}
agent1's action: 1
agent2's action: 1
reward is:  -92.5



joint_action is:  {'agent1': 2, 'agent2': 1}
agent1's action: 2
agent2's action: 1
reward is:  -125.0



Passed Parallel API test


### Ray

In [6]:
ray.init()

RuntimeError: Maybe you called ray.init twice by accident? This error can be suppressed by passing in 'ignore_reinit_error=True' or by calling 'ray.shutdown()' prior to 'ray.init()'.

In [ ]:
alg_name = "DQN"
ModelCatalog.register_custom_model("pa_model", TorchModelV2)

In [5]:
def env_creator():
        env = TrafficEnvironment(params[kc.SIMULATION_PARAMETERS])
        return env

Output()

action is:  0

reward is:  -87.0

action is:  2

reward is:  -153.0

...

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1        |
|    ep_rew_mean      | -104     |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 0        |
|    time_elapsed     | 9        |
|    total_timesteps  | 8        |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 101      |
|    n_updates        | 7        |
----------------------------------


action is:  1

reward is:  -87.0

action is:  1

reward is:  -87.0

In [ ]:
register_env("TrafficEnvironment", lambda config: PettingZooEnv(env_creator()))

In [6]:
test_env = PettingZooEnv(env_creator())
obs_space = test_env.observation_space
act_space = gym.spaces.Discrete(3)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
config = (
        DQNConfig()
        .environment(env="TrafficEnvironment")
        .rollouts(num_rollout_workers=1, rollout_fragment_length=30)
        .training(
            train_batch_size=200,
            hiddens=[],
            dueling=False,
            model={"custom_model": "pa_model"},
            optimizer={
            "adam": {
                "lr": 0.01,  # learning rate
                "momentum": 0.9,  # momentum (if applicable)
                # Add other optimizer parameters as needed
            }
        }
        )
        .multi_agent(
            policies={
                "player_0": (None, obs_space, act_space, {}),
                "player_1": (None, obs_space, act_space, {}),
            },
            policy_mapping_fn=(lambda agent_id, *args, **kwargs: agent_id),
        )
        .resources(num_gpus=int(os.environ.get("RLLIB_NUM_GPUS", "0")))
        .debugging(
            log_level="DEBUG"
        )  # TODO: change to ERROR to match pistonball example
        .framework(framework="torch")
        .exploration(
            exploration_config={
                # The Exploration class to use.
                "type": "EpsilonGreedy",
                # Config for the Exploration class' constructor:
                "initial_epsilon": 0.1,
                "final_epsilon": 0.0,
                "epsilon_timesteps": 100000,  # Timesteps over which to anneal epsilon.
            }
        )
    )


Output()

action is:  1

reward is:  -87.0

action is:  0

reward is:  -87.0

...

action is:  2

reward is:  -153.0

action is:  1

reward is:  -87.0

action is:  0

In [ ]:
config.environment(disable_env_checking=True)

In [ ]:
tune.run(
        alg_name,
        name="DQN",
        stop={"timesteps_total": 10000000 if not os.environ.get("CI") else 50000},
        checkpoint_freq=10,
        config=config.to_dict(),
    )